<h1>Búsquedas Informadas</h1></th></tr></tbody></table>



Este notebook contiene alguno de los problemas ya hechos en el de búsquedas no informadas y algunos nuevos, además de incluir en cada clase la heurística


In [1]:
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.optimize as opt
import time
from search import *

#### 1. Problema de los cántaros

In [2]:
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.optimize as opt
import time
from search import *

In [3]:
class Cantaros(Problem):

    def __init__(self, initial=(0,0), goal=(2,0)):
        super().__init__(initial, goal)
    
    def actions(self, state):
        c4, c3 = state
        possible_actions = []
        if c4 < 4:
            possible_actions.append("LLENAR_C4")
        if c3 < 3:
            possible_actions.append("LLENAR_C3")
        if c3 < 3 and c4 > 0:
            possible_actions.append("TRASVASAR_4_3")
        if c4 < 4 and c3 > 0:
            possible_actions.append("TRASVASAR_3_4")
        if c4 > 0:
            possible_actions.append("VACIAR_C4")
        if c3 > 0:
            possible_actions.append("VACIAR_C3")
        return possible_actions
    
    def result(self, state, action):
        c4, c3 = state
        if action == "LLENAR_C4":
            return (4, c3)
        elif action == "LLENAR_C3":
            return (c4, 3)
        elif action == "VACIAR_C4":
            return (0, c3)
        elif action == "TRASVASAR_4_3":
            espacio = 3 - c3
            cantidad = min(c4, espacio)
            return (c4 - cantidad, c3 + cantidad)
        elif action == "TRASVASAR_3_4":
            espacio = 4 - c4
            cantidad = min(espacio, c3)
            return (c4 + cantidad, c3 - cantidad)
        return state
    def goal_test(self,state):
        return self.goal == state
    def h(self, node):
        return abs(node.state[0] - self.goal[0]) + abs(node.state[1] - self.goal[1])

In [4]:
c = Cantaros()

print("Búsqueda en Anchura (BFS):")
start = time.time()
res_bfs = breadth_first_graph_search(c)
print("Movimientos cantaros:",res_bfs.solution())
print("Solucion bfs:", res_bfs)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(res_bfs.solution())}")

print("\nBúsqueda A*:")
start = time.time()
res_astar = astar_search(c)
print("Movimientos cantaros:",res_astar.solution())
print("Solucion astar:", res_astar)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(res_astar.solution())}")

Búsqueda en Anchura (BFS):
Movimientos cantaros: ['LLENAR_C3', 'TRASVASAR_3_4', 'LLENAR_C3', 'TRASVASAR_3_4', 'VACIAR_C4', 'TRASVASAR_3_4']
Solucion bfs: <Node (2, 0)>
Tiempo: 0.0000s. Pasos: 6

Búsqueda A*:
Movimientos cantaros: ['LLENAR_C3', 'TRASVASAR_3_4', 'LLENAR_C3', 'TRASVASAR_3_4', 'VACIAR_C4', 'TRASVASAR_3_4']
Solucion astar: <Node (2, 0)>
Tiempo: 0.0000s. Pasos: 6


#### 2. Empiezas con la secuencia ABABAEC, o en general cualquier secuencia formada por las letras A, B, C y E.

#### Puedes transformar esta secuencia utilizando las siguientes igualdades: AC = E, AB = BC, BB = E, Ex = x para cualquier x

#### Por ejemplo:

#### ABBC puede transformarse en AEC
#### luego en AC
#### y finalmente en E

#### El objetivo es producir la secuencia E.

In [5]:
class Secuencia(Problem):

    def __init__(self, initial="ABABAEC", goal="E"):
        super().__init__(initial, goal)

    def actions(self, state):
        actions = []

        for i in range(len(state) - 1):
            par = state[i:i+2]

            if par == "AC":
                actions.append(("AC_E", i))

            elif par == "AB":
                actions.append(("AB_BC", i))

            elif par == "BB":
                actions.append(("BB_E", i))

        #Ex = x
            elif par[0] == "E":
                actions.append(("E_X", i))

        return actions
    def result(self, state, action):
        regla, i = action

        if regla == "AC_E":
            return state[:i] + "E" + state[i+2:]

        if regla == "AB_BC":
            return state[:i] + "BC" + state[i+2:]

        if regla == "BB_E":
            return state[:i] + "E" + state[i+2:]

        elif regla == "E_X":
        
            return state[:i] + state[i+1:]

    def goal_test(self,state):
        return state == self.goal
    def h(self, node):
        return node.state.count("A") + node.state.count("B") + node.state.count("C")

In [6]:
s = Secuencia()

print("Búsqueda en Anchura (BFS):")
start = time.time()
sol_bfs = breadth_first_graph_search(s)

print("Movimientos secuencia:", sol_bfs.solution())
print("Solucion bfs:", sol_bfs)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(sol_bfs.solution())}")

print("\nBúsqueda A*:")
start = time.time()
sol_astar = astar_search(s)

print("Movimientos secuencia:", sol_astar.solution())
print("Solucion astar:", sol_astar)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(sol_astar.solution())}")

Búsqueda en Anchura (BFS):
Movimientos secuencia: [('AB_BC', 2), ('BB_E', 1), ('E_X', 1), ('AC_E', 0), ('E_X', 0), ('E_X', 1), ('AC_E', 0)]
Solucion bfs: <Node E>
Tiempo: 0.0000s. Pasos: 7

Búsqueda A*:
Movimientos secuencia: [('E_X', 5), ('AC_E', 4), ('AB_BC', 2), ('BB_E', 1), ('E_X', 1), ('AC_E', 0), ('E_X', 0)]
Solucion astar: <Node E>
Tiempo: 0.0000s. Pasos: 7


#### 3. Hay seis cajas de cristal en fila, cada una con un candado. Cada una de las cinco primeras cajas contiene una llave que abre la siguiente caja en la fila; la última caja contiene un plátano.  Tú tienes la llave de la primera caja y quieres conseguir el plátano.

In [7]:
class Cajas(Problem):
    def __init__(self,initial = (0,0,0,0,0,0), goal = (1,1,1,1,1,1)):
        super().__init__(initial, goal)
    def actions(self, state):
        
        for i in range(len(state)):
            if i == 0 and state[i] == 0:
                return ["Abrir caja 1"]
            elif state[i - 1] == 1 and state[i] ==0 and i == 1:
                return ["Abrir caja 2"] 
            elif state[i - 1] == 1 and state[i] ==0 and i == 2:
                return ["Abrir caja 3"] 
            elif state[i - 1] == 1 and state[i] ==0 and i == 3:
                return ["Abrir caja 4"] 
            elif state[i - 1] == 1 and state[i] ==0 and i == 4:
                return ["Abrir caja 5"] 
            elif state[i - 1] == 1 and state[i] ==0 and i == 5:
                return ["Abrir caja 6"]
        
    
    def result(self, state, action):
        if action == "Abrir caja 1":
            return (1,0,0,0,0,0)
        
        elif action == "Abrir caja 2":
            return (1,1,0,0,0,0)
        
        elif action == "Abrir caja 3":
            return (1,1,1,0,0,0)
        
        elif action == "Abrir caja 4":
            return (1,1,1,1,0,0)
        
        elif action == "Abrir caja 5":
            return (1,1,1,1,1,0)
        
        elif action == "Abrir caja 6":
            return (1,1,1,1,1,1)
        return state
    def goal_test(self,state):
        return self.goal == state
        
    def h(self, node): 
        cont = 0
        for i in range(len(node.state)):
            if self.goal[i] != node.state[i]:
                cont += 1
        return cont

In [8]:
cj = Cajas()

print("Búsqueda en Anchura (BFS):")
start = time.time()
res_bfs = breadth_first_graph_search(cj)
print("Movimientos cajas:",res_bfs.solution())
print("Solucion bfs:", res_bfs)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(res_bfs.solution())}")

print("\nBúsqueda A*:")
start = time.time()
res_astar = astar_search(cj)
print("Movimientos cajas:",res_astar.solution())
print("Solucion astar:", res_astar)
print(f"Tiempo: {time.time()-start:.4f}s. Pasos: {len(res_astar.solution())}")

Búsqueda en Anchura (BFS):
Movimientos cajas: ['Abrir caja 1', 'Abrir caja 2', 'Abrir caja 3', 'Abrir caja 4', 'Abrir caja 5', 'Abrir caja 6']
Solucion bfs: <Node (1, 1, 1, 1, 1, 1)>
Tiempo: 0.0000s. Pasos: 6

Búsqueda A*:
Movimientos cajas: ['Abrir caja 1', 'Abrir caja 2', 'Abrir caja 3', 'Abrir caja 4', 'Abrir caja 5', 'Abrir caja 6']
Solucion astar: <Node (1, 1, 1, 1, 1, 1)>
Tiempo: 0.0000s. Pasos: 6
